In [ ]:
from typing import Dict, Any
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

# 워크플로 단계 정의
class WorkflowStep:
    GREETING = "GREETING"
    PROCESSING = "PROCESSING"

# 그래프 상태 정의
class GraphState(BaseModel):
    name: str = Field(default="", description="시용자 이름")
    greeting: str = Field(default="", description="생성된 인사말")
    processed_message: str = Field(default="", description="처리된 최종 메세지")

# 첫 번째 노드 함수
def generate_greeting(state: GraphState) -> Dict[str, Any]:
    name = state.name or "아무개"
    greeting = f"안녕하세요, {name}님!"
    print(f"[Generate_greeting] 인사말 생성: {greeting}")
    return {"greeting":greeting}

# 두 번째 노드: 인사말을 처리하고 최종 메세지 생성
def process_message(state: GraphState) -> Dict[str, Any]:
    greeting = state.greeting
    processed_message = f"{greeting} LangGraph에 오신 것을 환영합니다!"
    print(f"[Process_message] 최종 메세지: {processed_message}")
    return {"processed_message":processed_message}

# 그래프 생성
def create_hello_graph():
    workflow = StateGraph(GraphState)
    # 노드 추가
    workflow.add_node(WorkflowStep.GREETING, generate_greeting)
    workflow.add_node(WorkflowStep.PROCESSING, process_message)
    # 시작점 설정
    workflow.add_edge(START, WorkflowStep.GREETING)
    # 노드 간 에지 추가
    workflow.add_edge(WorkflowStep.GREETING, WorkflowStep.PROCESSING)
    workflow.add_edge(WorkflowStep.PROCESSING, END)

    # 그래프 컴파일
    app = workflow.compile()
    return app

def main():
    print("===Hello LangGraph===\n")
    app = create_hello_graph()

    initial_state = GraphState(name='Dohy', greeting="", processed_message="")
    print("초기 상태:", initial_state.model_dump())
    print("\n--- 그래프 실행 시작---")

    # 그래프 실행
    final_state = app.invoke(initial_state)

    print("---그래프 실행 종료---\n")
    print("최종 상태: ", final_state)
    print(f"\n결과 메세지: {final_state['processed_message']}")
    # ASCII로 그래프 출력
    app.get_graph().print_ascii()

if __name__=="__main__":
    main()

===Hello LangGraph===

초기 상태: {'name': 'Dohy', 'greeting': '', 'processed_message': ''}

--- 그래프 실행 시작---
[Generate_greeting] 인사말 생성: 안녕하세요, Dohy님!
[Process_message] 최종 메세지: 안녕하세요, Dohy님! LangGraph에 오신 것을 환영합니다!
---그래프 실행 종료---

최종 상태:  {'name': 'Dohy', 'greeting': '안녕하세요, Dohy님!', 'processed_message': '안녕하세요, Dohy님! LangGraph에 오신 것을 환영합니다!'}

결과 메세지: 안녕하세요, Dohy님! LangGraph에 오신 것을 환영합니다!
+-----------+  
| __start__ |  
+-----------+  
       *       
       *       
       *       
 +----------+  
 | GREETING |  
 +----------+  
       *       
       *       
       *       
+------------+ 
| PROCESSING | 
+------------+ 
       *       
       *       
       *       
  +---------+  
  | __end__ |  
  +---------+  
b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x99\x00\x00\x01M\x08\x02\x00\x00\x00\x9c\xe6Z`\x00\x00\x10\x00IDATx\x9c\xec\x9d\x07|\x14\xc5\xdb\xc7\xe7\xee\x92k\xe9\xbd\x93\nJI\xa1\x83\xf4\x12\x9a\x84\x00\x01\xe9\xa1\'\xd2\xc1 /\n\n"E\x91&E:"E\x10\r\x04\x02\xfe\x89H\x9

In [12]:
from typing import Dict, Any, Literal
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import random

# 그래프 상태 정의: 워크플로 전체에서 공유되는 데이터 구조
class EmotionBotState(BaseModel):
    user_message: str = Field(default="", description="사용자 입력 메세지")
    emotion: str = Field(default="", description="분석된 감정")
    response: str = Field(default="", description="최종 응답 메세지")

# LangChain LLM 초기화: 감정 분석에 사용할 AI 모델 설정
llm = ChatOpenAI(model='gpt-5-mini', max_completion_tokens=500)

# LLM 기반 감정 분석 노드: 첫 번째 처리 단계
def analyze_emotion(state: EmotionBotState)->Dict[str,Any]:
    message = state.user_message
    print(f"LLM 감정 분석중: '{message}'")
    messages=[
        SystemMessage(content="당신은 감정 분석 전문가입니다. 사용자의 메세지를 분석하여 'positive','negative','neutral' 중 하나로 감정을 분류해주세요." \
        "답변은 반드시 하나의 단어만 출력하세요."),
        HumanMessage(content=f"다음 메세지의 감정을 분석해주세요: '{message}'"),
    ]

    response = llm.invoke(messages)
    emotion = response.content.strip().lower()
   
    # 유효성 검사
    if emotion not in  ['positive', 'negative', 'neutral']:
        emotion = 'neutral'

    print(f"LLM 감정 분석 결과: {emotion}")
    return {'emotion':emotion}

# 긍정적 응답 생성
def generate_positive_response(state: EmotionBotState)->Dict[str, Any]:
    responses = ['정말 좋은 소식이네요!', '기분이 좋으시군요!', '멋지네요!']
    return {'response': random.choice(responses)}

# 부정적 응답 생성
def generate_negative_response(state: EmotionBotState)->Dict[str,Any]:
    responses = ['힘든 시간이시군요, 괜찮아요.', '마음이 아프시겠어요.','더 좋은 날이 올 거에요.']
    return {'response': random.choice(responses)}

# 중립적 응답 생성
def generate_neutral_response(state: EmotionBotState)->Dict[str,Any]:
    responses=['감사해요! 더 자세히 말씀해주세요.','이해했어요. 다른 도룸이 필요하면 말씀하세요!','흥미로운 주제네요!']
    return {"response":random.choice(responses)}

# 조건부 라우팅 함수: 감정 분석 결과에 따라 다음 노드 결정
def route_by_emotion(state: EmotionBotState) -> Literal["Positive_response","Negative_response","Neutral_response"]:
    emotion = state.emotion
    print(f"라우팅: {emotion}")

    if emotion == 'positive':
        return "Positive_response"
    elif emotion == "negative":
        return "Negative_response"
    else:
        return "Neutral_response"
    
# 그래프 생성 함수: 전체 워크플로 구성
def create_emotion_bot_graph():
    workflow = StateGraph(EmotionBotState)

    # 노드 추가: 각 처리 단계를 그래프에 등록
    workflow.add_node("Analyze_emotion", analyze_emotion)
    workflow.add_node("Positive_response", generate_positive_response)
    workflow.add_node("Negative_response", generate_negative_response)
    workflow.add_node("Neutral_response", generate_neutral_response)

    # 시작 에지 설정: 워크플로의 진입점 정의
    workflow.add_edge(START, "Analyze_emotion")
    # 조건부 에지 설정: 동적 라우팅 구현
    workflow.add_conditional_edges(
        "Analyze_emotion",
        route_by_emotion,
        {
            "Positive_response":"Positive_response",
            "Negative_response":"Negative_response",
            "Neutral_response":"Neutral_response"
        }
    )

    # 종료 에지 설정: 각 응답 노드에서 워크플로 종료
    workflow.add_edge("Positive_response",END)
    workflow.add_edge("Negative_response",END)
    workflow.add_edge("Neutral_response",END)

    return workflow.compile()

def main():
    print('===감정 분석 챗봇 테스트===')
    app = create_emotion_bot_graph()

    test_cases=[
        "오늘 정말 기분이 좋아요!",
        "너무 슬프고 힘들어요...",
        "날씨가 어떤가요?"
    ]

    for i,message in enumerate(test_cases, 1):
        print(f"테스트 {i}: '{message}'")
        state = EmotionBotState(user_message=message)
        result = app.invoke(state)
        print(f"응답: {result['response']}\n")

    # 그래프 시각화
    mermaid_png = app.get_graph().draw_mermaid_png()
    with open("./02_Conditional_routing.png",'wb') as f:
        f.write(mermaid_png)


if __name__=="__main__":
    main()

===감정 분석 챗봇 테스트===
테스트 1: '오늘 정말 기분이 좋아요!'
LLM 감정 분석중: '오늘 정말 기분이 좋아요!'
LLM 감정 분석 결과: positive
라우팅: positive
응답: 정말 좋은 소식이네요!

테스트 2: '너무 슬프고 힘들어요...'
LLM 감정 분석중: '너무 슬프고 힘들어요...'
LLM 감정 분석 결과: negative
라우팅: negative
응답: 마음이 아프시겠어요.

테스트 3: '날씨가 어떤가요?'
LLM 감정 분석중: '날씨가 어떤가요?'
LLM 감정 분석 결과: neutral
라우팅: neutral
응답: 감사해요! 더 자세히 말씀해주세요.



In [19]:
# 체크포인터를 사용한 상태 관리
from typing import Dict, Any
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import json

# 그래프 상태 정의
class MemoryBotState(BaseModel):
    user_message: str = Field(default="", description="사용자 입력 메세지")
    user_name: str = Field(default="", description="사용자 이름")
    user_preferences: Dict[str, Any] = Field(default_factory=dict, description="사용자 선호도")
    response: str = Field(default="", description="최종 응답")

# LangChain LLM 초기화
llm = ChatOpenAI(model='gpt-5-mini')

# 메세지 처리 노드: 메모리 로드/저장은 checkpointer가 담당하므로 여기선 불필요
def process_message(state: MemoryBotState)->Dict[str,Any]:
    message = state.user_message
    user_name = state.user_name
    preferences = state.user_preferences.copy()

    system_prompt=f"""
    당신은 사용자의 정보를 기억하는 메모리 봇입니다.
    현재 기억하는 정보:
    - 사용자 이름: {user_name if user_name else '모름'}
    - 좋아하는 것: {preferences.get("likes",[])}
    - 싫어하는 것: {preferences.get("dislikes",[])}
    사용자 메세지를 분석하여 다음 JSON 형식으로 응답하세요.
    {{
        "response": "사용자에게 줄 응답메세지",
        "new_name": "새로 알게된 이름 (없으면 null)",
        "new_likes": ["새로 알게 된 좋아하는 것들"],
        "new_dislikes": ["새로 알게 된 싫어하는 것들"]
    }}
    """

    messages = [SystemMessage(content=system_prompt),HumanMessage(content=message)]

    response = llm.invoke(messages)
    result = json.loads(response.content)

    # 새로운 정보 업데이트
    if result.get("new_name"):
        user_name = result['new_name']

    if result.get("new_likes"):
        preferences.setdefault("likes",[]).extend(result['new_likes'])

    if result.get("new_dislikes"):
        preferences.setdefault("dislikes",[]).extend(result['new_dislikes'])

    bot_response = result.get("response","죄송해요, 이해하지 못했어요.")
    return{
        "response":bot_response,
        "user_name":user_name,
        "user_preferences":preferences
    }

# 메모리 봇 그래프 생성: InMemorySave 통합
def create_memory_bot_graph():
    # InMemorySaver로 자동 메모리 관리
    checkpointer = InMemorySaver()
    workflow = StateGraph(MemoryBotState)

    workflow.add_node("process_message",process_message)

    workflow.add_edge(START,"process_message")
    workflow.add_edge("process_message",END)

    return workflow.compile(checkpointer=checkpointer)

def main():
    print("=== InMemorySaver 메모리 봇 테스트 ===")
    app = create_memory_bot_graph()
    thread_id = "dohy_123"
    conversations=[
        "안녕!",
        "내 이름은 Dohy야",
        "육회를 좋아해",
        "해산물은 싫어해",
        "내 이름이 뭐라고?",
        "내가 좋아하는 것과 싫어하는 것은?"
    ]

    for i,message in enumerate(conversations,1):
        print(f"[{i}] 사용자: {message}")

        # InMemorySaver 사용 시 Config 설정
        config = {"configurable":{"thread_id":thread_id}}
        result = app.invoke({"user_message":message},config)

        print(f"[{i}] 챗봇: {result['response']}")
        print(f"메모리: 이름={result.get('user_name',"없음")},"
              f"좋아하는 것={result.get('user_preferences',{})}\n")

    mermaid_png = app.get_graph().draw_mermaid_png()
    with open("./03_InMemorySaver.png",'wb') as f:
        f.write(mermaid_png)
if __name__=="__main__":
    main()

=== InMemorySaver 메모리 봇 테스트 ===
[1] 사용자: 안녕!
[1] 챗봇: 안녕하세요! 반가워요. 무엇을 도와드릴까요?
메모리: 이름=,좋아하는 것={}

[2] 사용자: 내 이름은 Dohy야
[2] 챗봇: 알겠어요, Dohy님. 이름을 기억할게요!
메모리: 이름=Dohy,좋아하는 것={}

[3] 사용자: 육회를 좋아해
[3] 챗봇: 알겠어요 — 육회를 좋아하시는군요. 기억해둘게요!
메모리: 이름=Dohy,좋아하는 것={'likes': ['육회']}

[4] 사용자: 해산물은 싫어해
[4] 챗봇: 알겠어요. 해산물을 싫어하시는 걸 기억할게요.
메모리: 이름=Dohy,좋아하는 것={'likes': ['육회'], 'dislikes': ['해산물']}

[5] 사용자: 내 이름이 뭐라고?
[5] 챗봇: 당신의 이름은 Dohy입니다.
메모리: 이름=Dohy,좋아하는 것={'likes': ['육회'], 'dislikes': ['해산물']}

[6] 사용자: 내가 좋아하는 것과 싫어하는 것은?
[6] 챗봇: 기억하고 있는 내용은 다음과 같습니다. 좋아하는 것: 육회. 싫어하는 것: 해산물.
메모리: 이름=Dohy,좋아하는 것={'likes': ['육회'], 'dislikes': ['해산물']}



In [21]:
from typing import Dict,Any, Literal
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
import random

# 그래프 상태 정의
class GuessGameState(BaseModel):
    target_number:int = Field(default=0, description="맞춰야 할 숫자")
    user_guess: int = Field(default=0, description="사용자 추측")
    attempts: int = Field(default=0, description="시도 횟수")
    max_attempts: int = Field(default=5, description="최대 시도 횟수")
    game_status: str = Field(default="playing", description="게임 상태")
    response: str = Field(default="", description="응답 메세지")

# 게임 노드 설정
def game_setup(state:GuessGameState) -> Dict[str,Any]:
    target = random.randint(1,50)
    print("Game Start")

    # 상태 업데이트: 딕셔너리로 변경할 필드만 반환
    return{
        "target_number":target,
        "game_status":"playing",
        "response":f"1~50 사이의 숫자를 맞춰보세요. (최대 {state.max_attempts}회)",
        "attempts":0
    }

# 사용자 추측 노드
def user_guess(state: GuessGameState) -> Dict[str,Any]:
    guess = input("입력: ")
    print(f"[{state.attempts+1}번째 시도] 추측: {guess}")

    return {"user_guess":int(guess), "attempts":state.attempts+1}

# 추측 확인 노드
def check_guess(state: GuessGameState) -> Dict[str,Any]:
    target = state.target_number
    guess = state.user_guess
    attempts = state.attempts

    print(f"[check_guess] {guess} (시도: {attempts}회)")

    # 게임 상태에 따른 분기 처리
    if guess == target:
        print("정답!")
        return {
            "game_status":"won",
            "response":f"정답! {guess}를 {attempts}번 만에 맞췄습니다!",
        }
    elif attempts >= state.max_attempts:
        print("시도 횟수 초과")
        return{
            "game_status":"lost",
            "response":f"게임 종료! 정답은 {target}이었습니다."
        }

    else:
        hint = "더 큰 수" if guess<target else "더 작은 수"
        remaining = state.max_attempts - attempts
        print(f"계속 진행: {hint}")
        return{
            "game_status":"playing",
            "response": f"{guess}는 틀렸습니다. {hint}를 시도해보세요! (남은 기회: {remaining}회)"
        }
    
# 조건부 라우팅 함수
def route_game(state:GuessGameState)->Literal["continue","end"]:
    print(f"라우팅 체크: 상태={state.game_status}, 시도={state.attempts}")
    if state.game_status == "playing":
        return "continue"
    else:
        return "end"

# 루프 워크플로 그래프 생성
def create_guess_game_graph():
    # StateGraph 초기화: 상태 스키마 지정
    workflow = StateGraph(GuessGameState)

    workflow.add_node("setup", game_setup)
    workflow.add_node("guess", user_guess)
    workflow.add_node("check", check_guess)

    workflow.add_edge(START, "setup")
    workflow.add_edge("setup", "guess")
    workflow.add_edge("guess","check")

    # 조건부 에지: 동적 라우팅으로 루프 구현
    workflow.add_conditional_edges(
        "check", # 소스 노드
        route_game, # 라우팅 함수
        {
            "continue":"guess", # 게임 계속 → 다시 추측
            "end":END,
        }
    )

    return workflow.compile()

def main():
    print("===루프 워크플로 예제===\n")
    app = create_guess_game_graph()

    # 그래프 실행
    initial_state = GuessGameState(max_attempts=5)
    result = app.invoke(initial_state)

    print(f"\n최종 결과: {result['response']}")
    print(f"게임 상태: {result['game_status']}")
    print(f"총 시도: {result['attempts']}회")

    mermaid_png = app.get_graph().draw_mermaid_png()
    with open("./04_Loop_workflow.png",'wb') as f:
        f.write(mermaid_png)
if __name__=="__main__":
    main()

===루프 워크플로 예제===

Game Start
[1번째 시도] 추측: 25
[check_guess] 25 (시도: 1회)
계속 진행: 더 큰 수
라우팅 체크: 상태=playing, 시도=1
[2번째 시도] 추측: 30
[check_guess] 30 (시도: 2회)
계속 진행: 더 작은 수
라우팅 체크: 상태=playing, 시도=2
[3번째 시도] 추측: 27
[check_guess] 27 (시도: 3회)
계속 진행: 더 큰 수
라우팅 체크: 상태=playing, 시도=3
[4번째 시도] 추측: 29
[check_guess] 29 (시도: 4회)
계속 진행: 더 작은 수
라우팅 체크: 상태=playing, 시도=4
[5번째 시도] 추측: 28
[check_guess] 28 (시도: 5회)
정답!
라우팅 체크: 상태=won, 시도=5

최종 결과: 정답! 28를 5번 만에 맞췄습니다!
게임 상태: won
총 시도: 5회


In [23]:
from typing import Dict, Any
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
import time
import random

# State 클래스 정의
class DashboardState(BaseModel):
    user_location:str = "서울"
    weather_data: Dict[str, Any] = {}
    news_data: Dict[str, Any] = {}
    stock_data: Dict[str,Any] = {}
    dashboard_report: str = ""
    start_time: float = 0.0

# 코디네이터 노드
def coordinator(state: DashboardState)->Dict[str,Any]:
    print(f"대시보드 생성 시작 - 위치: {state.user_location}")
    return {"start_time": time.time()}

# 병렬 실행 노드1: 날씨 데이터 수집
def weather_checker(state: DashboardState)->Dict[str,Any]:
    print("날씨 확인 중...")
    time.sleep(random.uniform(1.0, 2.0))

    weather_info={
        "location": state.user_location,
        "condition": "맑음",
        "temperature": 22,
        "humidity": 65
    }

    print(f"날씨: {weather_info['condition']}, {weather_info['temperature']}℃")
    return {"weather_data":weather_info}

# 병렬 실행 노드2: 뉴스 데이터 수집
def news_fetcher(state: DashboardState) -> Dict[str,Any]:
    print("뉴스 수집 중...")
    time.sleep(random.uniform(1.0, 2.0))

    news_info={
        "articles":[
            {"title":"AI 기술 발전 소식", "summary":"AI 분야 새로운 혁신"},
            {"title":"경제 동향 분석", "summary":"글로벌 경제 전망"}
        ],
        "count":2
    }

    print(f"뉴스 {news_info['count']}개 수집 완료")
    return {'news_data': news_info}

# 병렬 실행 노드3: 주식 데이터 분석
def stock_analyzer(state:DashboardState)->Dict[str,Any]:
    print("주식 분석 중...")
    time.sleep(random.uniform(2.0, 3.0))

    stock_info={
        "KOSPI":{"price": 5656, "change":-12.5},
        "NASDAQ":{"price":27454, "change":-1}
    }

    print("주식 분석 완료")
    return {"stock_data": stock_info}

# 집계 노드: 모든 병렬 작업 완료 후 실행
def aggregator(state: DashboardState)->Dict[str,Any]:
    print("리포트 생성 중...")
    parallel_time = time.time() - state.start_time

    report = f"""
    대시보드 리포트
    날씨: {state.weather_data.get('condition',"N/A")} {state.weather_data.get('temperature',"N/A")}℃
    뉴스: {state.news_data.get('count',0)}개 기사
    주식: KOSPI {state.stock_data.get('KOSPI',{}).get('price','N/A')}
    실행시간: {parallel_time:.1f}초
    """

    print(f"대시보드 완료 ({parallel_time:.1f}초)")
    return {"dashboard_report": report}

def create_graph():
    workflow = StateGraph(DashboardState)

    workflow.add_node("coordinator",coordinator)
    workflow.add_node("weather",weather_checker)
    workflow.add_node("news",news_fetcher)
    workflow.add_node("stock",stock_analyzer)
    workflow.add_node("aggregator",aggregator)
    # 병렬 실행 구조 정의
    workflow.add_edge(START, "coordinator")
    workflow.add_edge("coordinator", "weather")
    workflow.add_edge("coordinator", "news")
    workflow.add_edge("coordinator","stock")
    workflow.add_edge("weather","aggregator")
    workflow.add_edge("news","aggregator")
    workflow.add_edge("stock","aggregator")
    workflow.add_edge("aggregator",END)

    return workflow.compile()

def main():
    print("=== LangGraph 병렬 실행 예제 ===\n")
    app=create_graph()
    initial_state = DashboardState(user_location="대전")
    print("병렬 실행 시작")
    result = app.invoke(initial_state)

    print("\n최종 결과:")
    print(result['dashboard_report'])

    mermaid_png = app.get_graph().draw_mermaid_png()
    with open("./05_Parallel_workflow.png",'wb') as f:
        f.write(mermaid_png)

if __name__=="__main__":
    main()

=== LangGraph 병렬 실행 예제 ===

병렬 실행 시작
대시보드 생성 시작 - 위치: 대전
뉴스 수집 중...
주식 분석 중...
날씨 확인 중...
뉴스 2개 수집 완료
날씨: 맑음, 22℃
주식 분석 완료
리포트 생성 중...
대시보드 완료 (2.7초)

최종 결과:

    대시보드 리포트
    날씨: 맑음 22℃
    뉴스: 2개 기사
    주식: KOSPI 5656
    실행시간: 2.7초
    


In [ ]:
import httpx
from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain.chat_models import init_chat_model
from geopy.geocoders import Nominatim
import math

def calculator(expression:str)->str:
    """수학 계산을 수행합니다."""
    print(f"계산 요청: {expression}")
    try:
        expression = expression.replace("sqrt","math.sqrt")
        expression = expression.replace("sin", "math.sin")
        expression = expression.replace("cos","math.cos")

        result = eval(expression, {"__builtins__":{}, "math":math})
        return f"계산 결과: {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"

def get_weather(city_name:str)->dict:
    """도시 이름을 받아 해당 도시의 현재 날씨 정보를 반환합니다."""
    if city_name:
        latitude, longitude = get_coordinates(city_name)
    else:
        raise ValueError("City name must be provided to get weather information.")

    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"
    response = httpx.get(url)
    response.raise_for_status() #  HTTP 에러가 발생하면 예외 반환
    return response.json()

def get_coordinates(city_name:str) -> tuple[float, float]:
    """도시 이름을 받아 위도와 경도를 반환합니다."""
    geolocator = Nominatim(user_agent='weather_app')
    location = geolocator.geocode(city_name)
    if location:
        return location.latitude, location.longitude
    else:
        raise ValueError(f"Could not find coordinates for {city_name}")

def currency_converter(amount: float, from_currency: str, to_currency:str) -> str:
    """통화 간 환율을 계산합니다."""
    print(f"{amount} {from_currency}를 {to_currency}로 변환합니다.")
    rates = {("USD","KRW"): 1446.54, ("KRW","USD"):0.0009}
    rate_key = (from_currency.upper(), to_currency.upper())
    if rate_key in rates:
        rate = rates[rate_key]
        converted = amount * rate
        return f"{amount} {from_currency} = {converted:.2f} {to_currency}"
    return f"{amount} {from_currency} = {amount} {to_currency} (동일 통화)"


def should_continue(state: MessagesState): # MessagesState: 상태 업데이트를 덮어쓰지 않고, 이어쓰기
    print("\n--- 분기 결정 ---")
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        print(f"결정: 도구 호출 필요 ({len(last_message.tool_calls)}개)")
        return "tools"
    else:
        print("결정: 최종 응답으로 종료")
        return END

def create_call_model_function(model_with_tools):
    """model_with_tools를 클로저(함수 안에 함수를 정의하고 반환할 때, 안쪽 함수가 바깥 변수를 참조하는 함수)로 캡처하는 call_model 함수 생성"""

    def call_model(state: MessagesState):
        """LLM을 호출하여 응답을 생성하는 노드 함수"""
        last_message = state['messages'][-1]
        # 도구 실행 결과를 받았는지, 아니면 사용자 질문을 받았는지에 따라 분기
        if isinstance(last_message, ToolMessage):
            print("\n---모델 호출 (도구 결과 기반)---")
            # 도구 실행 결과가 길 수 있으므로 일부만 출력
            print(f"입력(도구 결과): {last_message.content[:300]}...")
        else:
            print("\n---모델 호출 (사용자 질문 기반)---")
            print(f"입력(사용자 메세지): {last_message.content}")

        # 모델을 호출하여 다음 행동을 결정하게 함
        response = model_with_tools.invoke(state['messages'])

        # 모델의 결정에 따라 로그 출력
        if response.tool_calls:
            print(f"모델의 판단: 도구 호출 → {response.tool_calls}")
        else:
            print(f"모델의 판단: 최종 답변 생성 → {response.content}")

        return {"messages":[response]}

    return call_model

def create_graph(model_with_tools, tool_node):
    """LLM 워크플로 그래프 생성"""
    workflow = StateGraph(MessagesState)

    call_model = create_call_model_function(model_with_tools)
    workflow.add_node("call_model",call_model)
    workflow.add_node("tools", tool_node)

    workflow.add_edge(START, "call_model")
    workflow.add_conditional_edges("call_model", should_continue, ["tools",END])
    workflow.add_edge("tools","call_model")

    return workflow.compile()

def llm_tool_call(query:str):
    """하나의 질문에 대해 전체 LLM 워크플로를 실행하고 로그를 출력합니다."""
    tools = [calculator, get_weather, currency_converter]
    tool_node = ToolNode(tools)
    model = init_chat_model("gpt-5-mini", model_provider="openai")
    model_with_tools = model.bind_tools(tools)

    print(f"질문: {query}")
    print("-"*50)

    app = create_graph(model_with_tools, tool_node)
    mermaid_png = app.get_graph().draw_mermaid_png()
    with open("./06_ToolNode.png",'wb') as f:
        f.write(mermaid_png)

    app.invoke({"messages": [HumanMessage(content=query)]})

    print("-"*50)
    print("처리 완료")
    print("="*50+"\n")

def main():
    print("=== LangGraph ToolNode 예제 (LLM 기반) ===\n")

    test_queries= [
        "2+3*4를 계산해줘.",
        "sqrt(2) * sin(0.5)를 계산해줘",
        "대전 날씨 어때?",
        "100달러를 원화로 바꿔줘",
        "sqrt(16)을 계산해줘",
        "뉴욕 날씨가 궁금해",
        "1000원을 달러로 환전해줘"
    ]

    print("\nLLM 기반 도구 호출 시작:")
    for query in test_queries:
        try:
            llm_tool_call(query)
        except Exception as e:
            print(f"'{query}' 처리 중 오류 발생: {e}")
            print("="*50 + "\n")


if __name__ == "__main__":
    main()

=== LangGraph ToolNode 예제 (LLM 기반) ===


LLM 기반 도구 호출 시작:
질문: 2+3*4를 계산해줘.
--------------------------------------------------

---모델 호출 (사용자 질문 기반)---
입력(사용자 메세지): 2+3*4를 계산해줘.
모델의 판단: 최종 답변 생성 → 2 + 3 × 4 = 14  
계산 과정: 3×4 = 12 → 2 + 12 = 14.

--- 분기 결정 ---
결정: 최종 응답으로 종료
--------------------------------------------------
처리 완료

질문: sqrt(2) * sin(0.5)를 계산해줘
--------------------------------------------------

---모델 호출 (사용자 질문 기반)---
입력(사용자 메세지): sqrt(2) * sin(0.5)를 계산해줘
모델의 판단: 도구 호출 → [{'name': 'calculator', 'args': {'expression': 'sqrt(2) * sin(0.5)'}, 'id': 'call_dv7mLckuakpF1Abx9G5XM9Qd', 'type': 'tool_call'}]

--- 분기 결정 ---
결정: 도구 호출 필요 (1개)
계산 요청: sqrt(2) * sin(0.5)

---모델 호출 (도구 결과 기반)---
입력(도구 결과): 계산 결과: 0.6780100988420897...
모델의 판단: 최종 답변 생성 → 계산 결과는 약 0.6780100988420897입니다.

--- 분기 결정 ---
결정: 최종 응답으로 종료
--------------------------------------------------
처리 완료

질문: 대전 날씨 어때?
--------------------------------------------------

---모델 호출 (사용자 질문 기반)---
입력(사용자 메세지): 대전 날씨 어때?
모델의

In [1]:
# 휴먼 인 더 루프
from typing import Literal
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

class AgentState(BaseModel):
    user_message: str = Field(default="", description="사용자 입력 작업")
    task_details: str = Field(default="", description="작업 상세 정보")
    response: str = Field(default="", description="응답 결과")

def get_llm_response_node(state: AgentState, llm):
    """LLM과 상호작용하여 응답을 생성하거나, 추가 정보를 요청하는 노드"""
    details = state.task_details

    if details:
        print(f"\n상세 정보를 바탕으로 작업 실행: '{details}'")
        prompt = f"다음 요청에 따라 보고서를 작성해주세요: {details}"

    else:
        task = state.user_message
        print(f"\n작업 실행: '{task}' 작업을 수행합니다...")
        prompt =f"'{task}' 작업을 수행하려고 합니다. 어떤 종류의 보고서가 필요한지, 구체적인 주제는 무엇인지 질문해주세요. 추가 정보가 필요하면, 반드시 응답의 마지막을 물음표('?')로 끝내주세요."
    response = llm.invoke(prompt).content

    print("--- LLM 응답 ---")
    print(response)
    print("----------------")

    return {'response':response, 'task_details':""}

def get_task_details_node(state:AgentState) -> AgentState:
    """LLM의 질문에 대한 사용자 답변을 입력받는 노드"""
    print("\nLLM의 질문에 답변해주세요.")
    user_input = input("답변: ")
    return {"task_details":user_input}

def check_llm_response(state:AgentState) -> Literal['get_details',"end"]:
    """LLM의 응답이 질문인지 확인하여 다음 단계를 결정합니다."""
    print("LLM 응답 분석 중...")
    if state.response.strip().endswith("?"):
        print("LLM이 추가 정보를 요청했습니다. 사용자 입력을 받습니다.")
        return "get_details"

    print("최종 보고서가 생성되었습니다. 워크플로를 종료합니다.")
    return 'end'

def create_graph():
    """Human-in-the-loop 워크플로 그래프를 생성합니다."""
    # 그래프 전체에서 사용할 LLM 모델을 초기화합니다.
    llm = init_chat_model(model='gpt-5-mini', model_provider='openai')
    def get_llm_response_with_llm(state):
        return get_llm_response_node(state, llm)

    workflow = StateGraph(AgentState)
    workflow.add_node("get_llm_response", get_llm_response_with_llm)
    workflow.add_node("get_details", get_task_details_node)

    workflow.add_edge(START, "get_llm_response")
    workflow.add_conditional_edges(
        "get_llm_response",
        check_llm_response,
        {
            "get_details":"get_details",
            "end":END
        }
    )
    workflow.add_edge("get_details","get_llm_response")
    return workflow.compile()

def main():
    print("===LangGraph Human-in-the-loop 간소화 예제 ===\n")
    app = create_graph()
    mermaid_png = app.get_graph().draw_mermaid_png()
    with open("./07_HITL.png",'wb') as f:
        f.write(mermaid_png)
    final_state = app.invoke(AgentState(user_message="블로그 글 작성"))
    print("\n--- 워크플로 종료 ---")
    print("최종 응답:")
    print(final_state['response'])


if __name__ == "__main__":
    main()

c:\Users\Dohy\Desktop\dohy\Agent_recent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


===LangGraph Human-in-the-loop 간소화 예제 ===


작업 실행: '블로그 글 작성' 작업을 수행합니다...
--- LLM 응답 ---
블로그 글 작성을 위해 몇 가지 정보를 여쭙겠습니다. 아래 항목들을 알려주실 수 있나요?

1. 글의 주제(또는 후보 주제)와 전달하고자 하는 핵심 메시지는 무엇인가요?  
2. 대상 독자(연령대, 직업, 관심사 등)와 이 글의 목적(인지 증대, 판매 유도, 정보 제공 등)은 무엇인가요?  
3. 원하는 글 길이(단어 수, 문단 수 또는 예상 읽는 시간)는 어떻게 되나요?  
4. 톤·스타일(친근한·전문적인·카주얼·격식 등) 선호는 무엇인가요?  
5. 포함해야 할 SEO 키워드나 반드시 넣어야 할 문구가 있나요?  
6. 글 구조나 형식 요구(목차 필요 여부, 소제목 개수, FAQ 포함, 요약문 필요 등)는 무엇인가요?  
7. 참고할 자료나 레퍼런스(경쟁 블로그 URL 등)가 있나요?  
8. 이미지·도표·인포그래픽 필요 여부와 사용 가능한 이미지 제공 여부(저작권 이슈 포함)는 어떻게 되나요?  
9. 글 끝에 넣을 CTA(구독, 상담 신청, 구매 유도 등)나 내부 링크 연결 대상이 있나요?  
10. 마감일과 초안/최종본, 수정 횟수에 대한 요구사항이 있나요?  
11. 게시 플랫폼(네이버 블로그, 티스토리, 워드프레스 등)과 HTML 삽입 등 포맷 요구가 있나요?  
12. 다루기 민감한 주제(의료·법률·금융 등)나 피해야 할 표현이 있나요?  
13. 필요하다면 제목 후보, 메타 설명(검색용), 소셜 미디어용 요약문도 함께 만들어 드릴까요?

위 내용 중 우선 순위가 있거나 추가로 알려주실 사항이 있다면 함께 알려주시겠어요?
----------------
LLM 응답 분석 중...
LLM이 추가 정보를 요청했습니다. 사용자 입력을 받습니다.

LLM의 질문에 답변해주세요.

상세 정보를 바탕으로 작업 실행: 'LLM에 대한 설명 3줄로 적어줘'
--- LLM 응답 ---
대형 언어 모델(LL

In [7]:
import httpx
from geopy.geocoders import Nominatim
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langchain.chat_models import init_chat_model
from typing import Literal
import json

def get_weather(city_name:str)->dict:
    """도시 이름을 받아 해당 도시의 현재 날씨 정보를 반환합니다."""
    if city_name:
        latitude, longitude = get_coordinates(city_name)
    else:
        raise ValueError("City name must be provided to get weather information.")

    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"
    response = httpx.get(url)
    response.raise_for_status() #  HTTP 에러가 발생하면 예외 반환
    return response.json()

def get_coordinates(city_name:str) -> tuple[float, float]:
    """도시 이름을 받아 위도와 경도를 반환합니다."""
    geolocator = Nominatim(user_agent='weather_app')
    location = geolocator.geocode(city_name)
    if location:
        return location.latitude, location.longitude
    else:
        raise ValueError(f"Could not find coordinates for {city_name}")

def create_weather_agent():
    """날씨 관련 질문을 처리하는 전문가 하위 그래프를 생성합니다."""
    model = init_chat_model("gpt-5-mini", model_provider="openai").bind_tools([get_weather])
    tool_node = ToolNode([get_weather])

    def call_model(state:MessagesState):
        return {"messages":[model.invoke(state["messages"])]}

    graph = StateGraph(MessagesState)
    graph.add_node("call_model", call_model)
    graph.add_node("tool_node", tool_node)

    graph.add_edge(START, "call_model")
    graph.add_conditional_edges(
        "call_model",
        lambda s: "tool_node" if s['messages'][-1].tool_calls else END,
        {"tool_node":"tool_node", END: END},
    )
    graph.add_edge("tool_node", "call_model")
    return graph.compile()

def router(state:MessagesState) -> Literal['weather_expert', 'general_agent']:
    query = state['messages'][-1].content.lower()
    if "날씨" in query or "기온" in query:
        print("라우팅 결정: 기상 전문가에게 위임")
        return "weather_expert"
    print("라우팅 결정: 일반 에이전트가 처리")
    return "general_agent"

def create_main_agent(weather_subgraph):
    """질문을 라우팅하고 처리하는 메인 에이전트 그래프를 생성합니다."""
    main_model = init_chat_model("gpt-5-mini", model_provider="openai")

    workflow = StateGraph(MessagesState)
    workflow.add_node(
        "general_agent", lambda s: {"messages": [main_model.invoke(s['messages'])]}
    )
    workflow.add_node("weather_expert", weather_subgraph)
    workflow.add_conditional_edges(
        START,
        router,
        {
            "weather_expert": "weather_expert",
            "general_agent": "general_agent"
        }
    )
    workflow.add_edge("general_agent",END)
    workflow.add_edge("weather_expert",END)
    return workflow.compile()

def main():
    print("=== LangGraph 하위 그래프 예제 (기상 전문가) ===\n")
    weather_agent = create_weather_agent()
    main_agent = create_main_agent(weather_agent)

    main_graph_image = main_agent.get_graph(xray=True).draw_mermaid_png()
    with open("08_SubGraph.png","wb") as f:
        f.write(main_graph_image)
    queries = ["대전 날씨 어때?", "잠은 몇 시간 자는 게 좋을까?"]
    for query in queries:
        print(f"\n--- 질문: {query} ---")
        result = main_agent.invoke({"messages":[HumanMessage(content=query)]})
        print(f"최종 답변: {result['messages'][-1].content}")
        print("-"*20)

if __name__ == "__main__":
    main()



=== LangGraph 하위 그래프 예제 (기상 전문가) ===


--- 질문: 대전 날씨 어때? ---
라우팅 결정: 기상 전문가에게 위임
최종 답변: 지금 대전 날씨(기준 시각: 2026-08-01 13:15 KST)  
- 날씨: 대체로 맑음  
- 기온: 33.5°C (많이 더움)  
- 풍속: 13.1 km/h, 방향 서북서(286°)  
- 강수: 현재 비는 없음

간단한 주의사항: 더우니 수분 자주 섭취하시고 장시간 야외 활동 시 그늘이나 휴식, 자외선 차단하세요.
--------------------

--- 질문: 잠은 몇 시간 자는 게 좋을까? ---
라우팅 결정: 일반 에이전트가 처리
최종 답변: 좋은 질문입니다. 권장 수면 시간은 나이에 따라 다르며 개인차가 큽니다. 아래는 일반적인 권장 범위와 함께 실용적인 팁입니다.

권장 수면 시간(일반 가이드)
- 신생아(0–3개월): 14–17시간
- 영아(4–11개월): 12–15시간
- 유아(1–2세): 11–14시간
- 미취학 아동(3–5세): 10–13시간
- 초등·학령기 아동(6–13세): 9–11시간
- 청소년(14–17세): 8–10시간
- 성인(18–64세): 7–9시간
- 고령자(65세 이상): 7–8시간

주의할 점
- 위 수치는 권장 범위로, 개인차가 있어 같은 나이라도 필요한 시간이 다를 수 있습니다.
- 수면의 질(깊은 잠 비율, 중간 각성 여부)과 규칙성(매일 같은 시간에 자고 일어나는 습관)이 중요합니다.
- 보통 낮 동안 깨어있고 활동적이며 기분이 안정되면 수면 시간이 적절하다는 신호입니다.

부족하거나 과도한 수면의 신호
- 부족: 낮 동안 집중력 저하, 피로, 기분 변화, 사고·사고 위험 증가
- 과다(지속적 9–10시간 이상): 우울증이나 기저질환, 수면 무호흡증 같은 문제가 있을 수 있으므로 평가 필요

간단한 수면 개선 팁
- 매일 같은 시간에 자고 같은 시간에 일어나기(주말도 크게 벗어나지 않기)
- 취침 전 30–60분은 휴식 루틴(독서, 명상) 만들기